In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import random
import math
import time
import re
from pyvi import ViTokenizer
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng device: {device}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Đang sử dụng device: cuda


In [2]:
# Các Token đặc biệt
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
PAD_TOKEN = '<pad>'
UNK_TOKEN = '<unk>'

class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.itos = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN, 3: UNK_TOKEN}
        self.stoi = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocabulary(self, sentence_list):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in sentence:
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        return [self.stoi[token] if token in self.stoi else self.stoi[UNK_TOKEN] for token in text]

# Hàm tiền xử lý
def tokenize_en(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-zA-Z0-9]+", r" ", text)
    return text.split()

def tokenize_vi(text):
    text = text.lower().strip()
    return ViTokenizer.tokenize(text).split()

class TranslationDataset(Dataset):
    def __init__(self, en_path, vi_path, num_samples=30000):
        # Đọc dữ liệu (giới hạn số lượng để train nhanh trên Colab)
        with open(en_path, 'r', encoding='utf-8') as f:
            self.en_sentences = f.readlines()[:num_samples]
        with open(vi_path, 'r', encoding='utf-8') as f:
            self.vi_sentences = f.readlines()[:num_samples]

        print(f"Đã nạp {len(self.en_sentences)} câu.")

        # Tokenize
        self.en_tokenized = [tokenize_en(sent) for sent in self.en_sentences]
        self.vi_tokenized = [tokenize_vi(sent) for sent in self.vi_sentences]

        # Build Vocab
        self.en_vocab = Vocabulary(freq_threshold=2)
        self.en_vocab.build_vocabulary(self.en_tokenized)

        self.vi_vocab = Vocabulary(freq_threshold=2)
        self.vi_vocab.build_vocabulary(self.vi_tokenized)

    def __len__(self):
        return len(self.en_sentences)

    def __getitem__(self, index):
        en_num = [self.en_vocab.stoi[SOS_TOKEN]] + self.en_vocab.numericalize(self.en_tokenized[index]) + [self.en_vocab.stoi[EOS_TOKEN]]
        vi_num = [self.vi_vocab.stoi[SOS_TOKEN]] + self.vi_vocab.numericalize(self.vi_tokenized[index]) + [self.vi_vocab.stoi[EOS_TOKEN]]
        return torch.tensor(en_num), torch.tensor(vi_num)

def collate_fn(batch):
    en_batch, vi_batch = [], []
    for en_item, vi_item in batch:
        en_batch.append(en_item)
        vi_batch.append(vi_item)

    en_batch = nn.utils.rnn.pad_sequence(en_batch, padding_value=0, batch_first=False) # shape: [seq_len, batch]
    vi_batch = nn.utils.rnn.pad_sequence(vi_batch, padding_value=0, batch_first=False)

    return en_batch, vi_batch

# Khởi tạo DataLoader
en_file = 'en_sents'
vi_file = 'vi_sents'
try:
    dataset = TranslationDataset(en_file, vi_file, num_samples=20000)
    train_loader = DataLoader(dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    print("Vocab Eng size:", len(dataset.en_vocab))
    print("Vocab Vie size:", len(dataset.vi_vocab))
except FileNotFoundError:
    print("Vui lòng sửa lại đường dẫn file English và Vietnamese cho khớp với tên file giải nén trong thư mục 'dataset/'.")

Đã nạp 20000 câu.
Vocab Eng size: 4097
Vocab Vie size: 3262


In [3]:
from torch.utils.data import random_split
total_size = len(dataset)
train_size = int(0.8 * total_size)
valid_size = total_size - train_size
train_dataset, valid_dataset = random_split(dataset, [train_size, valid_size])
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

print(f"Số lượng batch Train: {len(train_loader)}")
print(f"Số lượng batch Valid: {len(valid_loader)}")

Số lượng batch Train: 250
Số lượng batch Valid: 63


In [4]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src)) # [src_len, batch_size, emb_dim]
        outputs, hidden = self.rnn(embedded)
        # hidden (Context Vector): [1, batch_size, hid_dim]
        return hidden

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, dropout):
        super().__init__()
        self.hid_dim = hid_dim
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim + hid_dim, hid_dim) # Thêm Context vector vào đầu vào
        self.fc_out = nn.Linear(emb_dim + hid_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, context):
        # input: [batch_size] -> cần thêm chiều seq_len = 1
        input = input.unsqueeze(0) # [1, batch_size]
        embedded = self.dropout(self.embedding(input)) # [1, batch_size, emb_dim]

        emb_con = torch.cat((embedded, context), dim=2) # [1, batch_size, emb_dim + hid_dim]
        output, hidden = self.rnn(emb_con, hidden)

        output = torch.cat((embedded.squeeze(0), hidden.squeeze(0), context.squeeze(0)), dim=1)
        prediction = self.fc_out(output) # [batch_size, output_dim]
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [src_len, batch_size]
        # trg: [trg_len, batch_size]
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        context = self.encoder(src)
        hidden = context

        # input ban đầu là <sos>
        input = trg[0,:]

        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden, context)
            outputs[t] = output

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        return outputs

In [5]:
def translate_greedy(sentence, model, en_vocab, vi_vocab, device, max_len=50):
    model.eval()
    tokens = [SOS_TOKEN] + tokenize_en(sentence) + [EOS_TOKEN]
    src_indexes = [en_vocab.stoi.get(tok, en_vocab.stoi[UNK_TOKEN]) for tok in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        context = model.encoder(src_tensor)
        hidden = context

    trg_indexes = [vi_vocab.stoi[SOS_TOKEN]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden, context)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == vi_vocab.stoi[EOS_TOKEN]:
            break

    trg_tokens = [vi_vocab.itos[i] for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])


def translate_sampling(sentence, model, en_vocab, vi_vocab, device, temperature=1.0, max_len=50):
    model.eval()
    tokens = [SOS_TOKEN] + tokenize_en(sentence) + [EOS_TOKEN]
    src_indexes = [en_vocab.stoi.get(tok, en_vocab.stoi[UNK_TOKEN]) for tok in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        context = model.encoder(src_tensor)
        hidden = context

    trg_indexes = [vi_vocab.stoi[SOS_TOKEN]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden, context)

        # Áp dụng Temperature
        probs = F.softmax(output.squeeze(0) / temperature, dim=0)
        # Lấy mẫu ngẫu nhiên dựa trên phân phối xác suất
        pred_token = torch.multinomial(probs, 1).item()

        trg_indexes.append(pred_token)
        if pred_token == vi_vocab.stoi[EOS_TOKEN]:
            break

    trg_tokens = [vi_vocab.itos[i] for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])


def translate_beam_search(sentence, model, en_vocab, vi_vocab, device, beam_width=3, max_len=50):
    model.eval()
    tokens = [SOS_TOKEN] + tokenize_en(sentence) + [EOS_TOKEN]
    src_indexes = [en_vocab.stoi.get(tok, en_vocab.stoi[UNK_TOKEN]) for tok in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        context = model.encoder(src_tensor)
        hidden = context

    # Khởi tạo Beam: Mỗi phần tử là tuple (sequence, hidden_state, score)
    beam = [([vi_vocab.stoi[SOS_TOKEN]], hidden, 0.0)]
    completed_sentences = []

    for _ in range(max_len):
        candidates = []
        for seq, hid, score in beam:
            if seq[-1] == vi_vocab.stoi[EOS_TOKEN]:
                completed_sentences.append((seq, score))
                continue

            trg_tensor = torch.LongTensor([seq[-1]]).to(device)
            with torch.no_grad():
                output, new_hidden = model.decoder(trg_tensor, hid, context)

            # Tính log softmax để cộng dồn điểm (tránh bị lỗi underflow)
            log_probs = F.log_softmax(output.squeeze(0), dim=0)
            top_probs, top_idx = log_probs.topk(beam_width)

            for i in range(beam_width):
                new_seq = seq + [top_idx[i].item()]
                new_score = score + top_probs[i].item()
                candidates.append((new_seq, new_hidden, new_score))

        # Sắp xếp các candidates theo điểm và giữ lại top `beam_width`
        beam = sorted(candidates, key=lambda x: x[2], reverse=True)[:beam_width]

        # Dừng sớm nếu tất cả các câu trong beam đều đã kết thúc
        if all(seq[-1] == vi_vocab.stoi[EOS_TOKEN] for seq, _, _ in beam):
            break

    # Nếu chưa có câu nào hoàn thành, lấy câu tốt nhất hiện tại trong beam
    if not completed_sentences:
        best_seq = beam[0][0]
    else:
        # Normalize score theo độ dài câu để công bằng (Length Penalty)
        best_seq = max(completed_sentences, key=lambda x: x[1] / len(x[0]))[0]

    trg_tokens = [vi_vocab.itos[i] for i in best_seq]
    # Lọc bỏ SOS và EOS
    trg_tokens = [t for t in trg_tokens if t not in [SOS_TOKEN, EOS_TOKEN]]

    # Gom các từ lại bằng cách thay thế "_" của Pyvi thành khoảng trắng
    result = " ".join(trg_tokens).replace("_", " ")
    return result

In [6]:
from tqdm import tqdm
import torch

INPUT_DIM = len(dataset.en_vocab)
OUTPUT_DIM = len(dataset.vi_vocab)
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, DEC_DROPOUT)
model = Seq2Seq(enc, dec, device).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
TRG_PAD_IDX = dataset.vi_vocab.stoi[PAD_TOKEN]
criterion = nn.CrossEntropyLoss(ignore_index=TRG_PAD_IDX)


def train(model, iterator, optimizer, criterion, clip, epoch):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(iterator, total=len(iterator), desc=f"Epoch {epoch}", leave=False)
    for i, (src, trg) in enumerate(progress_bar):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
        progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})
    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)


            output = model(src, trg, 0)

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

N_EPOCHS = 100
PATIENCE = 5
MIN_DELTA = 0.001
best_valid_loss = float('inf')
counter = 0

print(f"Bắt đầu huấn luyện (Max {N_EPOCHS} epochs, Patience: {PATIENCE})...")

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train(model, train_loader, optimizer, criterion, 1, epoch)
    valid_loss = evaluate(model, valid_loader, criterion)

    print(f'Epoch: {epoch:02} | Train Loss: {train_loss:.3f} | Valid Loss: {valid_loss:.3f}')

    if valid_loss < (best_valid_loss - MIN_DELTA):
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best_model.pt')
        counter = 0
        print(f"  --> Cải thiện! Đã lưu mô hình (Valid Loss giảm)")
    else:
        counter += 1
        print(f"  --> Valid Loss không cải thiện ({counter}/{PATIENCE})")
        if counter >= PATIENCE:
            print(f"!!! EARLY STOPPING: Dừng huấn luyện tại Epoch {epoch} !!!")
            break

model.load_state_dict(torch.load('best_model.pt'))
print("Đã tải lại trọng số tối ưu nhất để chuẩn bị dịch.")

Bắt đầu huấn luyện (Max 100 epochs, Patience: 5)...


Epoch: 01 | Train Loss: 4.717 | Valid Loss: 4.246
  --> Cải thiện! Đã lưu mô hình (Valid Loss giảm)


Epoch: 02 | Train Loss: 3.594 | Valid Loss: 3.755
  --> Cải thiện! Đã lưu mô hình (Valid Loss giảm)


Epoch: 03 | Train Loss: 2.979 | Valid Loss: 3.538
  --> Cải thiện! Đã lưu mô hình (Valid Loss giảm)


Epoch: 04 | Train Loss: 2.545 | Valid Loss: 3.427
  --> Cải thiện! Đã lưu mô hình (Valid Loss giảm)


...

Epoch: 07 | Train Loss: 1.791 | Valid Loss: 3.394
  --> Valid Loss không cải thiện (1/5)


Epoch: 08 | Train Loss: 1.665 | Valid Loss: 3.368
  --> Valid Loss không cải thiện (2/5)


Epoch: 09 | Train Loss: 1.563 | Valid Loss: 3.451
  --> Valid Loss không cải thiện (3/5)


Epoch: 10 | Train Loss: 1.473 | Valid Loss: 3.486
  --> Valid Loss không cải thiện (4/5)


Epoch: 11 | Train Loss: 1.387 | Valid Loss: 3.528
  --> Valid Loss không cải thiện (5/5)
!!! EARLY STOPPING: Dừng huấn luyện tại Epoch 11 !!!
Đã tải lại trọng số tối ưu nhất để chuẩn bị dịch.


In [9]:
import nltk
from nltk.translate.bleu_score import corpus_bleu
from tqdm.notebook import tqdm

def calculate_bleu_score(model, dataset, en_vocab, vi_vocab, device, num_samples=500):
    model.eval()

    references = [] # Câu dịch chuẩn (Ground Truth)
    candidates = [] # Câu do mô hình dịch ra

    # Lấy ngẫu nhiên vài trăm câu trong tập dữ liệu để test
    test_indices = random.sample(range(len(dataset)), min(num_samples, len(dataset)))

    print(f"Đang tính điểm BLEU trên {len(test_indices)} câu mẫu...")

    for i in tqdm(test_indices, leave=False):
        # Lấy câu tiếng Anh gốc
        en_tokens = dataset.en_tokenized[i]
        en_sentence = " ".join(en_tokens)

        # Lấy câu tiếng Việt chuẩn (bỏ token <sos> và <eos>)
        vi_tokens = dataset.vi_tokenized[i]
        references.append([vi_tokens]) # NLTK yêu cầu references là list of list

        # Lấy câu do mô hình dịch (dùng Greedy Search cho nhanh)
        translated_sentence = translate_greedy(en_sentence, model, en_vocab, vi_vocab, device)
        # Chuyển câu dịch về dạng list các từ
        candidate_tokens = translated_sentence.split()
        candidates.append(candidate_tokens)

    # Tính toán BLEU score (Sử dụng trọng số đều cho cụm 1-từ đến 4-từ)
    score = corpus_bleu(references, candidates)

    return score * 100 # Chuyển sang thang điểm 100

# Gọi hàm để tính điểm
bleu_score = calculate_bleu_score(model, dataset, dataset.en_vocab, dataset.vi_vocab, device, num_samples=300)

print(f"\n🏆 Điểm BLEU của mô hình là: {bleu_score:.2f} / 100")
print("(Ghi chú: Trong dịch máy thực tế, BLEU > 30 là đã có thể hiểu được, > 50 là chất lượng cao, rất hiếm khi đạt được > 60-70 vì ngôn ngữ quá đa dạng).")

Đang tính điểm BLEU trên 300 câu mẫu...


  0%|          | 0/300 [00:00<?, ?it/s]


🏆 Điểm BLEU của mô hình là: 35.57 / 100
(Ghi chú: Trong dịch máy thực tế, BLEU > 30 là đã có thể hiểu được, > 50 là chất lượng cao, rất hiếm khi đạt được > 60-70 vì ngôn ngữ quá đa dạng).


In [10]:
def translate_greedy(sentence, model, en_vocab, vi_vocab, device, max_len=50):
    model.eval()
    tokens = [SOS_TOKEN] + tokenize_en(sentence) + [EOS_TOKEN]
    src_indexes = [en_vocab.stoi.get(tok, en_vocab.stoi[UNK_TOKEN]) for tok in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        context = model.encoder(src_tensor)
        hidden = context

    trg_indexes = [vi_vocab.stoi[SOS_TOKEN]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden, context)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        if pred_token == vi_vocab.stoi[EOS_TOKEN]:
            break

    trg_tokens = [vi_vocab.itos[i] for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])


def translate_sampling(sentence, model, en_vocab, vi_vocab, device, temperature=1.0, max_len=50):
    model.eval()
    tokens = [SOS_TOKEN] + tokenize_en(sentence) + [EOS_TOKEN]
    src_indexes = [en_vocab.stoi.get(tok, en_vocab.stoi[UNK_TOKEN]) for tok in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        context = model.encoder(src_tensor)
        hidden = context

    trg_indexes = [vi_vocab.stoi[SOS_TOKEN]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden, context)

        # Áp dụng Temperature
        probs = F.softmax(output.squeeze(0) / temperature, dim=0)
        # Lấy mẫu ngẫu nhiên dựa trên phân phối xác suất
        pred_token = torch.multinomial(probs, 1).item()

        trg_indexes.append(pred_token)
        if pred_token == vi_vocab.stoi[EOS_TOKEN]:
            break

    trg_tokens = [vi_vocab.itos[i] for i in trg_indexes]
    return " ".join(trg_tokens[1:-1])


def translate_beam_search(sentence, model, en_vocab, vi_vocab, device, beam_width=3, max_len=50):
    model.eval()
    tokens = [SOS_TOKEN] + tokenize_en(sentence) + [EOS_TOKEN]
    src_indexes = [en_vocab.stoi.get(tok, en_vocab.stoi[UNK_TOKEN]) for tok in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(1).to(device)

    with torch.no_grad():
        context = model.encoder(src_tensor)
        hidden = context

    # Khởi tạo Beam: Mỗi phần tử là tuple (sequence, hidden_state, score)
    beam = [([vi_vocab.stoi[SOS_TOKEN]], hidden, 0.0)]
    completed_sentences = []

    for _ in range(max_len):
        candidates = []
        for seq, hid, score in beam:
            if seq[-1] == vi_vocab.stoi[EOS_TOKEN]:
                completed_sentences.append((seq, score))
                continue

            trg_tensor = torch.LongTensor([seq[-1]]).to(device)
            with torch.no_grad():
                output, new_hidden = model.decoder(trg_tensor, hid, context)

            # Tính log softmax để cộng dồn điểm (tránh bị lỗi underflow)
            log_probs = F.log_softmax(output.squeeze(0), dim=0)
            top_probs, top_idx = log_probs.topk(beam_width)

            for i in range(beam_width):
                new_seq = seq + [top_idx[i].item()]
                new_score = score + top_probs[i].item()
                candidates.append((new_seq, new_hidden, new_score))

        # Sắp xếp các candidates theo điểm và giữ lại top `beam_width`
        beam = sorted(candidates, key=lambda x: x[2], reverse=True)[:beam_width]

        # Dừng sớm nếu tất cả các câu trong beam đều đã kết thúc
        if all(seq[-1] == vi_vocab.stoi[EOS_TOKEN] for seq, _, _ in beam):
            break

    # Nếu chưa có câu nào hoàn thành, lấy câu tốt nhất hiện tại trong beam
    if not completed_sentences:
        best_seq = beam[0][0]
    else:
        # Normalize score theo độ dài câu để công bằng (Length Penalty)
        best_seq = max(completed_sentences, key=lambda x: x[1] / len(x[0]))[0]

    trg_tokens = [vi_vocab.itos[i] for i in best_seq]
    # Lọc bỏ SOS và EOS
    trg_tokens = [t for t in trg_tokens if t not in [SOS_TOKEN, EOS_TOKEN]]

    # Gom các từ lại bằng cách thay thế "_" của Pyvi thành khoảng trắng
    result = " ".join(trg_tokens).replace("_", " ")
    return result

In [11]:
test_sentences = [
    "hello, how are you today?",
    "i love learning machine learning.",
    "the cat is sleeping on the table.",
]

print("=== SO SÁNH CÁC CHIẾN LƯỢC DECODING ===\n")
for sentence in test_sentences:
    print(f"Câu gốc (EN): {sentence}")

    # 1. Greedy Search
    greedy_res = translate_greedy(sentence, model, dataset.en_vocab, dataset.vi_vocab, device)
    print(f"-> Greedy:    {greedy_res.replace('_', ' ')}")

    # 2. Beam Search
    beam_res = translate_beam_search(sentence, model, dataset.en_vocab, dataset.vi_vocab, device, beam_width=3)
    print(f"-> Beam (k=3): {beam_res}")

    # 3. Sampling
    samp_res = translate_sampling(sentence, model, dataset.en_vocab, dataset.vi_vocab, device, temperature=0.8)
    print(f"-> Sampling:  {samp_res.replace('_', ' ')}")

    print("-" * 50)

=== SO SÁNH CÁC CHIẾN LƯỢC DECODING ===

Câu gốc (EN): hello, how are you today?
-> Greedy:    hôm nay bạn hôm nay bạn hôm nay
-> Beam (k=3): hôm nay bạn hôm nay bạn hôm nay
-> Sampling:  hôm nay bạn đang bạn hôm nay
--------------------------------------------------
Câu gốc (EN): i love learning machine learning.
-> Greedy:    tôi yêu một cái khác
-> Beam (k=3): tôi yêu những người khác
-> Sampling:  tôi yêu rất cẩn thận
--------------------------------------------------
Câu gốc (EN): the cat is sleeping on the table.
-> Greedy:    con mèo đang trên bàn bàn
-> Beam (k=3): con mèo đang trên bàn bàn
-> Sampling:  con mèo đang ở trên bàn
--------------------------------------------------
